# Remap hypnogram labels

This notebook allows you to harmonize sleep stage labels across a heterogeneous
PSG database, converting various scoring conventions to the standard AASM format
(`W`, `N1`, `N2`, `N3`, `R`).

**How to use:** Run all cells (Kernel → Restart & Run All), then interact with the widgets.

In [ ]:
try:
    import os
    import re
    import numpy as np
    import pandas as pd
    from pathlib import Path
    import ipywidgets as widgets
    from types import SimpleNamespace
    from ipyfilechooser import FileChooser
    from IPython.display import display, HTML
except ImportError as e:
    print("⚠️ Error:", e)
else:
    print("✅ Packages successfully imported!")

#%% --- CONSTANTS ---

STANDARD_LABELS   = {'W', 'N1', 'N2', 'N3', 'R'}
ACCEPTABLE_LABELS = STANDARD_LABELS | {'MT'}
STANDARD_OPTIONS  = ['W', 'N1', 'N2', 'N3', 'R', 'MT', '']

DEFAULT_MAPPING = {
    'W': 'W', 'N1': 'N1', 'N2': 'N2', 'N3': 'N3', 'R': 'R',
    '0': 'W', '1': 'N1', '2': 'N2', '3': 'N3', '4': 'N3', '5': 'R',
    '?': 'W',
    'MT': 'MT',
    'S1': 'N1', 'S2': 'N2', 'S3': 'N3', 'S4': 'N3',
    'WAKE': 'W', 'REM': 'R', 'SWS': 'N3',
}

#%% --- STATE ---

STATE = SimpleNamespace(
    folder_path=None,
    hypno_suffix=None,
    hypno_files=[],
    hypno_data={},
    failed_load=[],
    label_configs=None,
    config_labels=None,
    configs=None,
    suspicious_labels=None,
    mid_question_files=[],
    question_corrections={},
    remap_by_config=None,
    question_flags=None,
    output_suffix='_Hypnogram_remapped.txt',
)
STATE.ui = {}

#%% --- HELPER FUNCTIONS ---

def detect_hypno_files(folder, suffix):
    found = []
    for root, _, files in os.walk(folder):
        for fname in files:
            if fname.startswith('.'):
                continue
            if suffix:
                if suffix in fname:
                    found.append(os.path.join(root, fname))
            else:
                if fname.endswith('.txt'):
                    found.append(os.path.join(root, fname))
    return sorted(found)


def extract_file_id(filepath, suffix):
    stem = Path(filepath).stem
    if suffix:
        suffix_stem = Path(suffix).stem
        if stem.endswith(suffix_stem):
            return stem[:-len(suffix_stem)]
    return stem


def apply_remap(hypno, remap_dict):
    remapped = hypno.copy()
    for raw_label, new_label in remap_dict.items():
        if new_label:
            remapped[remapped == raw_label] = new_label
    return remapped


def get_unexpected_labels(unique_labels):
    return [l for l in unique_labels if l not in STANDARD_LABELS]


def suggest_label(raw_label):
    return DEFAULT_MAPPING.get(raw_label, DEFAULT_MAPPING.get(raw_label.upper(), ''))


def is_suspicious_label(label):
    if label in DEFAULT_MAPPING:
        return False
    if label.strip().isdigit() and int(label.strip()) > 5:
        return True
    if len(label) > 3 or re.search(r'[^A-Za-z0-9]', label):
        return True
    return False


def get_epoch_context(hypno, epoch_idx, n_ctx=3):
    start = max(0, epoch_idx - n_ctx)
    end   = min(len(hypno), epoch_idx + n_ctx + 1)
    parts = []
    for i in range(start, end):
        parts.append(f"[{hypno[i]}]" if i == epoch_idx else str(hypno[i]))
    return " ".join(parts)


def print_in_scrollable_box(text, height=250, font_size="12px"):
    display(HTML(
        f'<pre style="overflow-y:scroll; height:{height}px; border:1px solid #ccc; '
        f'padding:8px; font-size:{font_size};">{text}</pre>'
    ))


def _show_btn_in(out_widget, btn_key):
    btn = STATE.ui.get(btn_key)
    try:
        out_widget.clear_output()
        if btn is not None:
            with out_widget:
                display(btn)
    except Exception:
        pass


def invalidate_after(stage):
    if stage == 'scan':
        STATE.remap_by_config      = None
        STATE.question_corrections = {}
        # out_question is handled directly by on_scan (shows btn or "no mid-?" message)
        _show_btn_in(out_remap,  'run_remap')
        _show_btn_in(out_save,   'run_save')
        _show_btn_in(out_verify, 'run_verify')
        try: out_end.clear_output()
        except Exception: pass
    elif stage == 'question':
        STATE.remap_by_config = None
        _show_btn_in(out_remap,  'run_remap')
        _show_btn_in(out_save,   'run_save')
        _show_btn_in(out_verify, 'run_verify')
        try: out_end.clear_output()
        except Exception: pass
    elif stage == 'remap':
        _show_btn_in(out_save,   'run_save')
        _show_btn_in(out_verify, 'run_verify')
        try: out_end.clear_output()
        except Exception: pass
    elif stage == 'save':
        _show_btn_in(out_verify, 'run_verify')

#%% --- SECTION HEADERS ---

section1 = widgets.HTML('''
<hr style="height:4px; background-color:black; border:none;">
<h2>1. Select your data folder</h2>
<p>Select the folder containing your database. The tool will automatically detect all
hypnogram files matching the specified suffix.
<br>&#x2022; Choose your data folder using the file browser below.
<br>&#x2022; Enter the suffix common to all your hypnogram files
    (e.g. <code>_Hypnogram_Export.txt</code>).
<br>&#x2022; Leave the suffix empty to detect all <code>.txt</code> files in the folder.
<br>&#x2022; Click <b>Scan for hypnograms</b> to detect files and extract label configurations.</p>
''')

section2 = widgets.HTML('''
<hr style="height:4px; background-color:black; border:none;">
<h2>2. Review mid-recording &#x27;?&#x27; epochs</h2>
<p>Some hypnograms contain <code>?</code> labels in the middle of the recording (outside
the first/last 10 epochs). These often correspond to missed or ambiguous epochs.
<br>&#x2022; <b>Recommendation:</b> verify these epochs directly in your sleep scoring
    software before assigning a label here.
<br>&#x2022; Use the navigation below to inspect each epoch with its surrounding context.
<br>&#x2022; Assign a label and click <b>Apply</b>. The correction is applied in memory immediately.
<br>&#x2022; When done, click <b>Corrections done &#x2014; proceed to Section 3</b> to reset
    downstream sections.</p>
''')

section3 = widgets.HTML('''
<hr style="height:4px; background-color:black; border:none;">
<h2>3. Define remapping rules</h2>
<p>For each label configuration found in your database, assign the target standard AASM label.
<br>&#x2022; Click a configuration to unfold its list of original labels.
<br>&#x2022; Use the dropdowns to assign a standard label
    (<code>W</code>, <code>N1</code>, <code>N2</code>, <code>N3</code>, <code>R</code>).
    Suggestions are pre-filled.
<br>&#x2022; Suspicious labels are highlighted in red with &#xb1;5 epochs of surrounding context.
<br>&#x2022; If <code>?</code> labels appear here, they are boundary epochs (first/last ~10 epochs)
    not reviewed in Section 2 &#x2014; map them to <code>W</code>.
<br>&#x2022; Repeat for each configuration, then click <b>Save remapping</b>.</p>
''')

section4 = widgets.HTML('''
<hr style="height:4px; background-color:black; border:none;">
<h2>4. Preview and save remapped hypnograms</h2>
<p>&#x2022; Set the output file suffix (appended to the participant ID,
    e.g. <code>15_N1_Hypnogram_remapped.txt</code>).
<br>&#x2022; Click <b>Preview</b> to inspect the remapping before saving.
<br>&#x2022; Click <b>Save files</b> to write the remapped hypnograms next to the originals.
<br>If unexpected labels remain after remapping, a confirmation will be required before saving.</p>
''')

section5 = widgets.HTML('''
<hr style="height:4px; background-color:black; border:none;">
<h2>5. Verify remapping</h2>
<p>Reload the saved files and verify the remapping was applied correctly across the
full database.</p>
''')

#%% --- OUTPUT ZONES ---

out_scan     = widgets.Output()
out_configs  = widgets.Output()
out_question = widgets.Output()
out_remap    = widgets.Output()
out_save     = widgets.Output()
out_verify   = widgets.Output()
out_end      = widgets.Output()

#%% --- SECTION 1: FOLDER SELECTION & SCAN ---

chooser = FileChooser(os.getcwd())
chooser.title = "<b>Choose your data folder</b>"
chooser.show_only_dirs = True

suffix_box = widgets.Text(
    value='_Hypnogram_Export.txt',
    description='Hypnogram suffix:',
    placeholder='e.g. _Hypnogram_Export.txt  (leave empty for all .txt)',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='520px')
)

btn_scan = widgets.Button(
    description='Scan for hypnograms', button_style='primary', icon='search',
    layout=widgets.Layout(width='220px')
)


def on_scan(_=None):
    out_scan.clear_output()
    out_configs.clear_output()

    folder = chooser.selected_path
    if not folder:
        with out_scan:
            print("⚠️ Please select a folder first.")
        return

    suffix = suffix_box.value.strip()
    STATE.folder_path  = folder
    STATE.hypno_suffix = suffix

    with out_scan:
        print(f"📁 Folder: {folder}")
        if suffix:
            print(f"🔍 Suffix filter: '{suffix}'")
        else:
            print("🔍 No suffix filter (all .txt files)")

        hypno_files = detect_hypno_files(folder, suffix)
        STATE.hypno_files = hypno_files

        if not hypno_files:
            print("\n⚠️ No hypnogram files found. Check your suffix or folder.")
            return

        print(f"\n✅ Found {len(hypno_files)} hypnogram file(s).")

        hypno_data     = {}
        failed         = []
        question_flags = {}
        output         = ""
        dyn_out        = widgets.Output()
        display(dyn_out)

        for fpath in hypno_files:
            file_id  = extract_file_id(fpath, suffix)
            output  += f"Loading: {file_id}\n"
            with dyn_out:
                dyn_out.clear_output(wait=True)
                print_in_scrollable_box(output)
            try:
                hypno = np.loadtxt(fpath, dtype=str).astype('<U10')
                hypno_data[file_id] = hypno
                q_idx       = np.where(hypno == '?')[0]
                problematic = [int(i) for i in q_idx if i > 10 and i < len(hypno) - 10]
                if problematic:
                    question_flags[file_id] = (problematic, len(hypno))
            except Exception as e:
                output += f"  ❌ Error: {e}\n"
                failed.append((fpath, str(e)))

        dyn_out.clear_output()

        STATE.hypno_data     = hypno_data
        STATE.failed_load    = failed
        STATE.question_flags = question_flags

        print(f"Loaded: {len(hypno_data)}  |  Failed: {len(failed)}")

        configs_dict = {}
        for file_id, hypno in hypno_data.items():
            key = frozenset(np.unique(hypno).tolist())
            configs_dict.setdefault(key, [])
            configs_dict[key].append(file_id)

        configs       = list(configs_dict.keys())
        config_labels = [
            f"config. {i+1} (n={len(configs_dict[c])})"
            for i, c in enumerate(configs)
        ]

        STATE.label_configs = configs_dict
        STATE.configs       = configs
        STATE.config_labels = config_labels

        print(f"\n📊 {len(configs)} unique label configuration(s) found:")
        for i, cfg in enumerate(configs):
            print(f"  Config. {i+1} (n={len(configs_dict[cfg])}): {sorted(cfg)}")

        suspicious = {}
        for file_id, hypno in hypno_data.items():
            for label in np.unique(hypno):
                if is_suspicious_label(label):
                    suspicious.setdefault(label, [])
                    suspicious[label].append(file_id)
        STATE.suspicious_labels = suspicious

        if suspicious:
            print(f"\n⚠️ Suspicious labels detected ({len(suspicious)} label(s)):")
            for label, file_ids in suspicious.items():
                fid = file_ids[0]
                idx = int(np.where(hypno_data[fid] == label)[0][0])
                ctx = get_epoch_context(hypno_data[fid], idx, n_ctx=3)
                print(f"  '{label}': {len(file_ids)} file(s). "
                      f"Example context ({fid}, ep.{idx}): {ctx}")
            print("  → These labels will appear highlighted in Section 3.")
        else:
            print("\n✅ No suspicious labels detected.")

        STATE.mid_question_files = list(question_flags.keys())

        if question_flags:
            print(f"\n⚠️ Mid-recording '?' found in {len(question_flags)} file(s):")
            for file_id, (idxs, n) in question_flags.items():
                print(f"  {file_id}: epoch(s) {idxs} / {n} total")
            print("  → Verify them in your scoring software (Section 2).")
        else:
            print("\n✅ No mid-recording '?' issues detected.")

        max_len = max((len(sorted(c)) for c in configs), default=0)
        data = {}
        for i, cfg in enumerate(configs):
            fids      = configs_dict[cfg]
            col       = f"config. {i+1}<br>(n={len(fids)})"
            ls        = sorted(cfg)
            data[col] = ls + [''] * (max_len - len(ls))

        if data:
            df_table = pd.DataFrame(data)
            df_table.index = pd.Index(range(1, max_len + 1), name='label #')
            with out_configs:
                out_configs.clear_output()
                display(HTML("<h4>Label configurations found:</h4>"))
                display(HTML(df_table.to_html(escape=False)))

        invalidate_after('scan')

        # Section 2 is always visible; update its content based on scan results
        if STATE.mid_question_files:
            with out_question:
                out_question.clear_output()
                display(STATE.ui['run_question'])
        else:
            with out_question:
                out_question.clear_output()
                display(HTML(
                    "<p>✅ No mid-recording '?' epochs found. Proceed to Section 3.</p>"
                ))


btn_scan.on_click(on_scan)

#%% --- SECTION 2: MID-RECORDING '?' NAVIGATION ---

btn_run_question = widgets.Button(
    description='Run mid-? correction widget', button_style='success',
    layout=widgets.Layout(width='250px')
)
STATE.ui['run_question'] = btn_run_question


def run_question_widget(_=None):
    with out_question:
        out_question.clear_output()

        if not STATE.mid_question_files:
            display(HTML("<p>✅ No mid-recording '?' to review.</p>"))
            return

        ctx_slider = widgets.IntSlider(
            value=5, min=1, max=20, step=1,
            description='Context (epochs):',
            style={'description_width': '140px'},
            layout=widgets.Layout(width='380px'),
            continuous_update=False
        )

        file_options = [
            (f"{fid}  ({len(STATE.question_flags[fid][0])} mid-'?' found)", i)
            for i, fid in enumerate(STATE.mid_question_files)
        ]
        file_dropdown = widgets.Dropdown(
            options=file_options, value=0,
            description='File:',
            style={'description_width': '40px'},
            layout=widgets.Layout(width='480px')
        )

        btn_prev    = widgets.Button(description='← Previous', layout=widgets.Layout(width='120px'))
        btn_next    = widgets.Button(description='Next →',     layout=widgets.Layout(width='120px'))
        nav_label   = widgets.HTML()
        context_out = widgets.Output()

        label_combo = widgets.Combobox(
            options=STANDARD_OPTIONS, value='',
            placeholder='Pick or type label…',
            ensure_option=False,
            layout=widgets.Layout(width='120px')
        )
        btn_apply    = widgets.Button(description='Apply', button_style='primary',
                                      layout=widgets.Layout(width='80px'))
        apply_out    = widgets.Output()
        progress_lbl = widgets.HTML()

        nav = {'file': 0, 'epoch': 0, 'updating': False}

        def current_file_id():
            return STATE.mid_question_files[nav['file']]

        def current_q_epochs():
            return STATE.question_flags[current_file_id()][0]

        def update_progress():
            corrected = len(STATE.question_corrections.get(current_file_id(), {}))
            progress_lbl.value = (
                f"<b>Progress:</b> {corrected} / {len(current_q_epochs())} corrected for this file"
            )

        def refresh_context():
            context_out.clear_output()
            fid      = current_file_id()
            q_epochs = current_q_epochs()
            ep_pos   = nav['epoch']
            epoch    = q_epochs[ep_pos]
            n_total  = STATE.question_flags[fid][1]

            nav_label.value = (
                f"<b>File: {fid}</b> — Epoch <b>{epoch}</b> / {n_total}"
                f"  &nbsp; [{ep_pos + 1} / {len(q_epochs)}]"
            )

            hypno = STATE.hypno_data[fid]
            n_ctx = ctx_slider.value
            start = max(0, epoch - n_ctx)
            end   = min(len(hypno), epoch + n_ctx + 1)

            rows = []
            for i in range(start, end):
                lbl     = hypno[i]
                is_curr = (i == epoch)
                is_corr = (i in STATE.question_corrections.get(fid, {}))
                if is_curr:
                    style = 'color:#dc3545; font-weight:bold;'
                    prefix, sfx = '>>> ', ' <<<'
                elif lbl == '?' and not is_corr:
                    style = 'color:#fd7e14;'
                    prefix, sfx = '    ', ''
                elif is_corr:
                    style = 'color:#198754;'
                    prefix, sfx = '    ', ' ✓'
                else:
                    style = ''
                    prefix, sfx = '    ', ''
                rows.append(
                    f'<tr style="{style}">'
                    f'<td style="padding:1px 14px; font-family:monospace;">{prefix}{i}</td>'
                    f'<td style="padding:1px 14px; font-family:monospace;">{lbl}{sfx}</td></tr>'
                )

            table = (
                '<table style="font-size:13px; border:1px solid #ddd; border-radius:4px;">'
                '<tr><th style="padding:2px 14px; text-align:left;">Epoch</th>'
                '<th style="padding:2px 14px; text-align:left;">Label</th></tr>'
                + ''.join(rows) + '</table>'
            )
            with context_out:
                display(HTML(table))

            label_combo.value = STATE.question_corrections.get(fid, {}).get(epoch, '')
            update_progress()

        def go_to(file_i, epoch_i):
            nav['file']     = file_i
            nav['epoch']    = epoch_i
            nav['updating'] = True
            file_dropdown.value = file_i
            nav['updating'] = False
            refresh_context()

        def on_prev(_=None):
            ep = nav['epoch']
            if ep > 0:
                go_to(nav['file'], ep - 1)
            elif nav['file'] > 0:
                fi   = nav['file'] - 1
                last = len(STATE.question_flags[STATE.mid_question_files[fi]][0]) - 1
                go_to(fi, last)

        def on_next(_=None):
            q  = current_q_epochs()
            ep = nav['epoch']
            if ep < len(q) - 1:
                go_to(nav['file'], ep + 1)
            elif nav['file'] < len(STATE.mid_question_files) - 1:
                go_to(nav['file'] + 1, 0)

        def on_file_change(change):
            if nav['updating']:
                return
            nav['file']  = change['new']
            nav['epoch'] = 0
            refresh_context()

        def on_apply(_=None):
            apply_out.clear_output()
            new_label = label_combo.value.strip()
            if not new_label:
                with apply_out:
                    print("⚠️ Please select or type a label first.")
                return
            fid   = current_file_id()
            epoch = current_q_epochs()[nav['epoch']]
            STATE.hypno_data[fid][epoch] = new_label
            STATE.question_corrections.setdefault(fid, {})[epoch] = new_label
            with apply_out:
                print(f"✅ ep.{epoch} → '{new_label}'")
            refresh_context()
            on_next()

        btn_prev.on_click(on_prev)
        btn_next.on_click(on_next)
        btn_apply.on_click(on_apply)
        file_dropdown.observe(on_file_change, names='value')
        ctx_slider.observe(lambda _: refresh_context(), names='value')

        btn_done = widgets.Button(
            description='Corrections done — proceed to Section 3',
            button_style='success', icon='check',
            layout=widgets.Layout(width='310px')
        )
        done_out = widgets.Output()

        def on_done(_=None):
            done_out.clear_output()
            invalidate_after('question')
            with done_out:
                total = sum(len(v) for v in STATE.question_corrections.values())
                print(f"✅ {total} correction(s) applied to data in memory.")
                print("   Section 3 has been reset. Run the remapping widget.")

        btn_done.on_click(on_done)

        display(
            ctx_slider, file_dropdown, nav_label,
            widgets.HBox([btn_prev, btn_next]),
            context_out,
            widgets.HBox([label_combo, btn_apply]),
            apply_out, progress_lbl,
            widgets.HTML('<hr style="margin:12px 0;">'),
            btn_done, done_out,
        )
        refresh_context()


btn_run_question.on_click(run_question_widget)

#%% --- SECTION 3: REMAPPING WIDGET ---

btn_run_remap = widgets.Button(
    description='Run label remapping widget', button_style='success',
    layout=widgets.Layout(width='240px')
)
STATE.ui['run_remap'] = btn_run_remap


def run_remap_widget(_=None):
    with out_remap:
        out_remap.clear_output()

        if not STATE.hypno_data:
            print("⚠️ Please scan your folder first (Section 1).")
            return

        # Recompute configs from current hypno_data (picks up Section 2 corrections)
        configs_dict = {}
        for file_id, hypno in STATE.hypno_data.items():
            key = frozenset(np.unique(hypno).tolist())
            configs_dict.setdefault(key, [])
            configs_dict[key].append(file_id)

        configs       = list(configs_dict.keys())
        config_labels = [
            f"config. {i+1} (n={len(configs_dict[c])})"
            for i, c in enumerate(configs)
        ]

        row_widgets_by_cfg = {}
        panels = []

        for cfg, cfg_label in zip(configs, config_labels):
            row_widgets_by_cfg[cfg_label] = {}
            sorted_labels = sorted(cfg)
            participants  = sorted(configs_dict[cfg])

            rows = []
            for raw_label in sorted_labels:
                suggested  = suggest_label(raw_label)
                is_suspect = is_suspicious_label(raw_label)

                combo = widgets.Combobox(
                    options=STANDARD_OPTIONS, value=suggested,
                    placeholder='Pick or type…',
                    ensure_option=False,
                    layout=widgets.Layout(width='120px')
                )
                color = '#dc3545' if is_suspect else 'inherit'
                lbl   = widgets.HTML(
                    f'<span style="font-family:monospace; color:{color}; '
                    f'width:80px; display:inline-block;">{raw_label}</span>'
                )
                arrow     = widgets.Label('→', layout=widgets.Layout(width='25px'))
                row_items = [lbl, arrow, combo]

                if is_suspect:
                    ctx_text = ''
                    for fid in sorted(configs_dict[cfg])[:1]:
                        hypno = STATE.hypno_data[fid]
                        idxs  = np.where(hypno == raw_label)[0]
                        if len(idxs) > 0:
                            idx      = int(idxs[0])
                            ctx_str  = get_epoch_context(hypno, idx, n_ctx=5)
                            ctx_text = (f"⚠️ Suspicious — context "
                                        f"({fid}, ep.{idx}): {ctx_str}")
                    row_items.append(widgets.HTML(
                        f'<small style="color:#dc3545; margin-left:8px;">{ctx_text}</small>'
                    ))

                rows.append(widgets.HBox(row_items))
                row_widgets_by_cfg[cfg_label][raw_label] = combo

            panel_items = [
                widgets.HTML(
                    f"<i>Participants ({len(participants)}): {', '.join(participants)}</i>"
                )
            ]

            # Boundary-? note: ? surviving to this stage are start/end epochs, not mid-recording
            if '?' in sorted_labels:
                panel_items.append(widgets.HTML(
                    '<div style="background:#fff3cd; border:1px solid #ffc107; '
                    'padding:6px 10px; border-radius:4px; margin:4px 0; font-size:12px;">'
                    'ℹ️ <b>Note:</b> <code>?</code> labels at this stage are boundary '
                    'epochs (first/last ~10 epochs of the recording) not reviewed in Section 2. '
                    'They are typically unscored — map them to <code>W</code>.</div>'
                ))

            panel_items.append(widgets.VBox(rows, layout=widgets.Layout(
                max_height='320px', overflow='auto',
                border='1px solid #ddd', padding='6px'
            )))

            panels.append(widgets.VBox(panel_items))

        acc = widgets.Accordion(children=panels)
        for i, lbl in enumerate(config_labels):
            acc.set_title(i, lbl)

        btn_save_remap = widgets.Button(
            description='Save remapping', button_style='success', icon='save'
        )
        save_remap_out = widgets.Output()

        def on_save_remap(_=None):
            remap         = {}
            non_aasm_warn = {}

            for cfg_label in config_labels:
                mapping = {
                    raw: combo.value.strip()
                    for raw, combo in row_widgets_by_cfg[cfg_label].items()
                }
                remap[cfg_label] = mapping
                bad = [v for v in mapping.values() if v and v not in ACCEPTABLE_LABELS]
                if bad:
                    non_aasm_warn[cfg_label] = sorted(set(bad))

            STATE.remap_by_config = remap
            invalidate_after('remap')

            with save_remap_out:
                save_remap_out.clear_output()
                print("✅ Remapping saved:")
                for cfg_label, mapping in remap.items():
                    print(f"\n  {cfg_label}:")
                    for raw, new in mapping.items():
                        flag = "  ⚠️ empty target!" if not new else ""
                        print(f"    '{raw}'  →  '{new}'{flag}")
                if non_aasm_warn:
                    print("\n⚠️ Non-AASM target labels detected:")
                    for cfg_label, labels in non_aasm_warn.items():
                        print(f"  {cfg_label}: {labels}")
                    print("  → Standard AASM labels: W, N1, N2, N3, R  (MT also accepted)")

        btn_save_remap.on_click(on_save_remap)
        display(acc, widgets.HBox([btn_save_remap]), save_remap_out)


btn_run_remap.on_click(run_remap_widget)

#%% --- SECTION 4: PREVIEW & SAVE ---

btn_run_save = widgets.Button(
    description='Run preview & save', button_style='success',
    layout=widgets.Layout(width='220px')
)
STATE.ui['run_save'] = btn_run_save


def run_save_widget(_=None):
    with out_save:
        out_save.clear_output()

        if STATE.remap_by_config is None:
            print("⚠️ Please define and save the remapping first (Section 3).")
            return

        cur_configs_dict = {}
        for file_id, hypno in STATE.hypno_data.items():
            key = frozenset(np.unique(hypno).tolist())
            cur_configs_dict.setdefault(key, [])
            cur_configs_dict[key].append(file_id)
        cur_configs = list(cur_configs_dict.keys())
        cur_labels  = [
            f"config. {i+1} (n={len(cur_configs_dict[c])})"
            for i, c in enumerate(cur_configs)
        ]
        file_id_to_cfg = {}
        for cfg, lbl in zip(cur_configs, cur_labels):
            for fid in cur_configs_dict[cfg]:
                file_id_to_cfg[fid] = lbl

        remapped_data      = {}
        unexpected_by_file = {}
        for file_id, hypno in STATE.hypno_data.items():
            cfg_label = file_id_to_cfg.get(file_id)
            if cfg_label is None or cfg_label not in STATE.remap_by_config:
                continue
            remapped = apply_remap(hypno, STATE.remap_by_config[cfg_label])
            remapped_data[file_id] = remapped
            unexpected = get_unexpected_labels(np.unique(remapped).tolist())
            if unexpected:
                unexpected_by_file[file_id] = unexpected

        has_unexpected = bool(unexpected_by_file)
        if has_unexpected:
            display(HTML(
                '<div style="background:#fff3cd; border:1px solid #ffc107; '
                'padding:10px; border-radius:4px; margin-bottom:8px;">'
                f'<b>⚠️ Warning:</b> {len(unexpected_by_file)} file(s) still have '
                'non-standard labels after remapping.</div>'
            ))
            for fid, labels in unexpected_by_file.items():
                display(HTML(
                    f"<p style='margin:2px 0 2px 16px'>&#x2022; <b>{fid}</b>: {labels}</p>"
                ))
            display(HTML(
                "<p>Go back to Section 3 to add rules, or confirm saving below.</p>"
            ))

        suffix_box_out = widgets.Text(
            value=STATE.output_suffix,
            description='Output suffix:',
            style={'description_width': '120px'},
            layout=widgets.Layout(width='420px')
        )
        btn_preview    = widgets.Button(description='Preview', button_style='info', icon='eye')
        btn_save_files = widgets.Button(
            description='Save files', button_style='success', icon='save',
            disabled=has_unexpected
        )
        confirm_check  = widgets.Checkbox(
            value=False,
            description='Confirm saving despite the warnings above',
            layout=widgets.Layout(width='440px')
        )

        if has_unexpected:
            def on_confirm(change):
                btn_save_files.disabled = not change['new']
            confirm_check.observe(on_confirm, names='value')
            display(confirm_check)

        preview_out    = widgets.Output()
        save_files_out = widgets.Output()
        display(widgets.HBox([suffix_box_out, btn_preview, btn_save_files]))
        display(preview_out, save_files_out)

        def on_preview(_=None):
            preview_out.clear_output()
            with preview_out:
                out_suffix = suffix_box_out.value.strip()
                print(f"Output suffix: '{out_suffix}'")
                print("\nRemapping rules:")
                for cfg_label, mapping in STATE.remap_by_config.items():
                    print(f"  {cfg_label}: {mapping}")
                print("\nLabel distribution after remapping (first 5 files):")
                for file_id, remapped in list(remapped_data.items())[:5]:
                    unique, counts = np.unique(remapped, return_counts=True)
                    dist = {k: int(v) for k, v in zip(unique.tolist(), counts.tolist())}
                    print(f"  {file_id}: {dist}")
                if len(remapped_data) > 5:
                    print(f"  ... and {len(remapped_data) - 5} more file(s).")

        def on_save_files(_=None):
            out_suffix = suffix_box_out.value.strip()
            if not out_suffix:
                with save_files_out:
                    print("⚠️ Please define an output suffix.")
                return

            STATE.output_suffix = out_suffix
            saved, failed_save  = [], []

            for fpath in STATE.hypno_files:
                file_id = extract_file_id(fpath, STATE.hypno_suffix)
                if file_id not in remapped_data:
                    continue
                out_path = Path(fpath).parent / (file_id + out_suffix)
                try:
                    np.savetxt(out_path, remapped_data[file_id], fmt='%s')
                    saved.append(str(out_path))
                except Exception as e:
                    failed_save.append((str(out_path), str(e)))

            with save_files_out:
                save_files_out.clear_output()
                print(f"✅ Saved {len(saved)} remapped hypnogram(s).")
                if saved:
                    print(f"  → Location: {Path(saved[0]).parent}")
                if failed_save:
                    print(f"❌ Failed to save {len(failed_save)} file(s):")
                    for p, e in failed_save:
                        print(f"  {p}: {e}")

            invalidate_after('save')

            if saved:
                with out_end:
                    out_end.clear_output()
                    display(HTML(
                        '<hr style="height:4px; background-color:black; border:none;">'
                        '<h2>✅ Done!</h2>'
                        '<p>Your remapped hypnograms are ready for downstream analyses '
                        '(spectral analysis, YASA, statistics…).</p>'
                        f'<p><b>Files saved with suffix:</b> <code>{out_suffix}</code></p>'
                        '<p><b>To load a remapped hypnogram in Python:</b></p>'
                        '<pre style="background:#f8f9fa; padding:8px; border-radius:4px;">'
                        'import numpy as np\n'
                        f'hypno = np.loadtxt("participantID{out_suffix}", dtype=str)'
                        '</pre>'
                    ))

        btn_preview.on_click(on_preview)
        btn_save_files.on_click(on_save_files)


btn_run_save.on_click(run_save_widget)

#%% --- SECTION 5: VERIFY ---

btn_run_verify = widgets.Button(
    description='Run verification', button_style='success',
    layout=widgets.Layout(width='200px')
)
STATE.ui['run_verify'] = btn_run_verify


def run_verify(_=None):
    with out_verify:
        out_verify.clear_output()

        folder = STATE.folder_path
        suffix = STATE.output_suffix

        if not folder or not suffix:
            print("⚠️ Please complete the previous steps first.")
            return

        saved_files = detect_hypno_files(folder, suffix)
        if not saved_files:
            print(f"⚠️ No files found with suffix '{suffix}'. Run Section 4 first.")
            return

        print(f"📂 Found {len(saved_files)} remapped file(s) with suffix '{suffix}'.\n")

        configs_after  = {}
        non_aasm_after = {}
        failed_after   = []

        for fpath in saved_files:
            file_id = extract_file_id(fpath, suffix)
            try:
                hypno = np.loadtxt(fpath, dtype=str).astype('<U10')
                key   = frozenset(np.unique(hypno).tolist())
                configs_after.setdefault(key, [])
                configs_after[key].append(file_id)
                non_aasm = [l for l in np.unique(hypno) if l not in ACCEPTABLE_LABELS]
                if non_aasm:
                    non_aasm_after[file_id] = non_aasm
            except Exception as e:
                failed_after.append((file_id, str(e)))

        n_before = len(STATE.configs) if STATE.configs is not None else '?'
        print(f"─── Before remapping: {n_before} configuration(s) ───")
        if STATE.configs is not None and STATE.label_configs is not None:
            for i, cfg in enumerate(STATE.configs):
                fids = STATE.label_configs[cfg]
                print(f"  Config. {i+1} (n={len(fids)}): {sorted(cfg)}")

        print(f"\n─── After remapping: {len(configs_after)} configuration(s) ───")
        for i, (cfg, fids) in enumerate(configs_after.items()):
            non_std = [l for l in sorted(cfg) if l not in ACCEPTABLE_LABELS]
            note    = f"  ⚠️ non-AASM: {non_std}" if non_std else ""
            print(f"  Config. {i+1} (n={len(fids)}): {sorted(cfg)}{note}")
        if len(configs_after) > 1:
            print("  (Multiple configurations are normal when some participants "
                  "are missing certain sleep stages, e.g. no N3 in insomnia patients.)")

        if not non_aasm_after:
            print("\n✅ Remapping successful! All files contain only AASM-standard "
                  "labels (W, N1, N2, N3, R, MT).")
        else:
            print(f"\n❌ {len(non_aasm_after)} file(s) still contain non-AASM labels. "
                  "Review your mapping rules in Section 3.")
            for fid, labels in non_aasm_after.items():
                print(f"  {fid}: {labels}")

        if failed_after:
            print(f"\n❌ Could not reload {len(failed_after)} file(s):")
            for fid, e in failed_after:
                print(f"  {fid}: {e}")


btn_run_verify.on_click(run_verify)

#%% --- MAIN LAYOUT ---

ui_layout = widgets.VBox([
    section1,
    chooser,
    suffix_box,
    widgets.HBox([btn_scan]),
    out_scan,
    out_configs,
    section2,
    out_question,
    section3,
    out_remap,
    section4,
    out_save,
    section5,
    out_verify,
    out_end,
])

display(ui_layout)